# 01 Explore the UCI concrete data

A short look at the real dataset behind the concrete emulator: distributions, strength versus
age, strength versus water-to-binder ratio, and how many unique mixes there are.
Run `uv run python -m formopt download` first.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from formopt.config import load_config
from formopt.data import INGREDIENTS, load_concrete

paths = load_config().abs_paths
df = load_concrete(paths.raw_dir / "concrete")
print(df.shape, "rows x cols;", df["mix_id"].nunique(), "unique mixes")
df.describe().T

## Distributions

Zeros in slag, fly ash and superplasticizer are real values (the mix has none), not missing data.

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(11, 8))
for ax, col in zip(axes.ravel(), [*INGREDIENTS, "age_days", "strength_mpa"], strict=True):
    ax.hist(df[col], bins=30, color="#0072B2")
    ax.set_title(col)
fig.tight_layout()

## Strength versus age

Strength rises with curing age. 28 days is the usual specification age.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(df["age_days"], df["strength_mpa"], s=8, alpha=0.4)
ax.set_xscale("log")
ax.set_xlabel("Age (days, log scale)")
ax.set_ylabel("Compressive strength (MPa)")
med = df.groupby("age_days")["strength_mpa"].median()
ax.plot(med.index, med.values, color="black", label="median by age")
ax.legend()

## Strength versus water-to-binder ratio (28 days)

The single strongest relationship in concrete: more water per unit of binder means weaker
concrete. Binder here is cement plus slag plus fly ash.

In [ ]:
d28 = df[df["age_days"] == 28].copy()
d28["wb"] = d28["water"] / (d28["cement"] + d28["slag"] + d28["fly_ash"])
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(d28["wb"], d28["strength_mpa"], s=10, alpha=0.6)
ax.set_xlabel("Water / binder")
ax.set_ylabel("28-day strength (MPa)")
print(
    "Spearman correlation:",
    round(d28[["wb", "strength_mpa"]].corr(method="spearman").iloc[0, 1], 3),
)

## Repeated mixes

The same recipe appears at several ages. A random row split would put the same mix in train
and test, so the emulator is validated with grouped cross-validation on `mix_id`.

In [ ]:
ages_per_mix = df.groupby("mix_id")["age_days"].nunique()
print(ages_per_mix.value_counts().sort_index())
print(
    "Share of rows whose mix appears at more than one age:",
    round(float(np.mean(df["mix_id"].map(ages_per_mix) > 1)), 3),
)